In [19]:
from pathlib import Path
import pandas as pd
import pybedtools
from tqdm import tqdm
from collections import defaultdict

In [153]:
ROOT = Path().cwd()
DATA = Path("../data").resolve()
FANTOM = DATA / "fantom_cat"
PREPROCESSING = Path("../preprocessing")
RESULTS = Path("../results")
HIMORNA_INDEX_BED = PREPROCESSING / "himorna_index-bed"
RNACHROM_INDEX_BED = PREPROCESSING / "rnachrom_index-bed"
PERMUTATION_RES = PREPROCESSING / "permutation_res"
PERMUTATION_RES_AFTER_FDR = RESULTS / "rnachrom_himorna-intersection"
DOCKER_R = Path("../docker_r")

In [21]:
[i for i in RNACHROM_INDEX_BED.glob("*/*ENSG00000282057*")][2]#

PosixPath('../preprocessing/rnachrom_index-bed/GRID-MDA_MB_231__HTB_26_ATCC_/GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057-sort.bed')

In [22]:
[pd.read_csv(i) for i in (RNACHROM_INDEX_BED / "GRID-MDA_MB_231__HTB_26_ATCC_").glob("GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057-sort.bed")]

[           chr1\t603522\t603544\t.\t.\t-
 0          chr1\t719426\t719448\t.\t.\t-
 1        chr1\t1260823\t1260846\t.\t.\t-
 2        chr1\t1303789\t1303811\t.\t.\t-
 3        chr1\t1547431\t1547454\t.\t.\t-
 4        chr1\t1654565\t1654587\t.\t.\t-
 ...                                  ...
 17852  chrY\t14074719\t14074742\t.\t.\t-
 17853  chrY\t18239734\t18239757\t.\t.\t-
 17854  chrY\t19537335\t19537358\t.\t.\t-
 17855  chrY\t23262864\t23262886\t.\t.\t-
 17856  chrY\t25567471\t25567493\t.\t.\t-
 
 [17857 rows x 1 columns]]

In [23]:
res_a = [pd.read_csv(i, sep="\t") for i in (RNACHROM_INDEX_BED / "GRID-MDA_MB_231__HTB_26_ATCC_").glob("GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057-sort.bed")][0]
res_a

,chr1,603522,603544,.,..1,-
0,chr1,719426,719448,.,.,-
1,chr1,1260823,1260846,.,.,-
2,chr1,1303789,1303811,.,.,-
3,chr1,1547431,1547454,.,.,-
4,chr1,1654565,1654587,.,.,-
...,...,...,...,...,...,...
17852,chrY,14074719,14074742,.,.,-
17853,chrY,18239734,18239757,.,.,-
17854,chrY,19537335,19537358,.,.,-
17855,chrY,23262864,23262886,.,.,-


In [24]:
res_a

,chr1,603522,603544,.,..1,-
0,chr1,719426,719448,.,.,-
1,chr1,1260823,1260846,.,.,-
2,chr1,1303789,1303811,.,.,-
3,chr1,1547431,1547454,.,.,-
4,chr1,1654565,1654587,.,.,-
...,...,...,...,...,...,...
17852,chrY,14074719,14074742,.,.,-
17853,chrY,18239734,18239757,.,.,-
17854,chrY,19537335,19537358,.,.,-
17855,chrY,23262864,23262886,.,.,-


In [25]:
to_rnachrom_example = (RNACHROM_INDEX_BED / "GRID-MDA_MB_231__HTB_26_ATCC_/GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057-sort.bed").absolute()
to_rnachrom_example

PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts/../preprocessing/rnachrom_index-bed/GRID-MDA_MB_231__HTB_26_ATCC_/GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057-sort.bed')

In [26]:
res_b = [pd.read_csv(i, sep="\t") for i in (HIMORNA_INDEX_BED / "H3K9ac").glob("H3K9ac-ENSG00000282057-sort.bed")][0]

In [36]:
to_himorna_example = (HIMORNA_INDEX_BED / "H3K9ac/H3K9ac-ENSG00000282057-sort.bed").absolute()
to_himorna_example

PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts/../preprocessing/himorna_index-bed/H3K9ac/H3K9ac-ENSG00000282057-sort.bed')

In [188]:
res_b

,chr1,1354819,1362691
0,chr1,1354819,1362691
1,chr1,1419332,1419526
2,chr1,1419332,1419526
3,chr1,1419332,1419526
4,chr1,1829123,1829724
...,...,...,...
16048,chrX,154045623,154047638
16049,chrX,154045623,154047638
16050,chrX,154493122,154493355
16051,chrX,154817293,154820964


In [37]:
#создание файла для распараллеливания
[i for i in RNACHROM_INDEX_BED.glob("*/*ENSG00000282057*")]

[PosixPath('../preprocessing/rnachrom_index-bed/HFFc6/HFFc6-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/rnachrom_index-bed/iMARGI-HFFc6/iMARGI-HFFc6-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/rnachrom_index-bed/GRID-MDA_MB_231__HTB_26_ATCC_/GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/rnachrom_index-bed/Red_C-fibroblasts/Red_C-fibroblasts-ENSG00000282057-sort.bed')]

In [38]:
[i for i in (HIMORNA_INDEX_BED).glob("*/*-ENSG00000282057-sort.bed")]

[PosixPath('../preprocessing/himorna_index-bed/H3K27ac/H3K27ac-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/H3K27me3/H3K27me3-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/H3K36me3/H3K36me3-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/H3K4me1/H3K4me1-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/H3K4me2/H3K4me2-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/H3K4me3/H3K4me3-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/H3K79me2/H3K79me2-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/H3K9ac/H3K9ac-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/H3K9me3/H3K9me3-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/H4K20me1/H4K20me1-ENSG00000282057-sort.bed'),
 PosixPath('../preprocessing/himorna_index-bed/methylation/methylation-ENSG000002820

In [39]:
unique_lncrna_rnachrom = set([i.absolute().stem.split("-")[1] for i in RNACHROM_INDEX_BED.glob("*/*") if i.stem != ".ipynb_checkpoints"])

In [40]:
len(unique_lncrna_rnachrom)

3189

In [41]:
unique_lncrna_himorna = set([i.absolute().stem.split("-")[1] for i in HIMORNA_INDEX_BED.glob("*/*") if i.stem != ".ipynb_checkpoints"])

In [42]:
len(unique_lncrna_himorna)

4434

In [43]:
len(unique_lncrna_himorna)

4434

In [46]:
unique_lncrna = unique_lncrna_rnachrom & unique_lncrna_himorna
len(unique_lncrna)

3182

In [129]:
output_tsv = Path("pairs.tsv")

with output_tsv.open("w") as out:
    for lncrna in tqdm(unique_lncrna):
        paths_to_rnachrom_data = [i.absolute() for i in RNACHROM_INDEX_BED.glob(f"*/*-{lncrna}-sort.bed")]
        paths_to_himorna_data = [i.absolute() for i in HIMORNA_INDEX_BED.glob(f"*/*-{lncrna}-sort.bed")]
        
        for lncA in paths_to_rnachrom_data:
            for lncB in paths_to_himorna_data:
                out.write(f"{lncA}\t{lncB}\n")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3182/3182 [02:24<00:00, 22.05it/s]


In [125]:
paths_to_himorna_data

[PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts/../preprocessing/himorna_index-bed/H3K27ac/H3K27ac-ENSG00000255328-sort.bed'),
 PosixPath('/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/scripts/../preprocessing/himorna_index-bed/methylation/methylation-ENSG00000255328-sort.bed')]

In [ ]:
#Запуск скрипта work/git_projects/fantom.deg-himorna.peaks/docker_r/permutation-regioner-03_07_25.R через parallel

In [9]:
from statsmodels.stats.multitest import multipletests

In [47]:
to_pairs_tsv = DOCKER_R / "pairs.tsv"
pair_tsv = pd.read_csv(to_pairs_tsv, sep="\t", header=None)
pair_tsv.head(2)

,0,1
0,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...
1,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...


In [45]:
for lncrna in unique_lncrna:
    to_txt = PERMUTATION_RES.glob(f"*{unique_lncrna}*.txt")
    if len(to_txt) !+

{'ENSG00000179818',
 'ENSG00000227014',
 'ENSG00000228221',
 'ENSG00000261777',
 'ENSG00000235663',
 'ENSG00000250131',
 'ENSG00000125804',
 'ENSG00000234028',
 'ENSG00000248015',
 'ENSG00000250001',
 'ENSG00000268670',
 'ENSG00000256571',
 'ENSG00000225194',
 'ENSG00000280355',
 'ENSG00000227039',
 'ENSG00000241388',
 'ENSG00000249096',
 'ENSG00000245812',
 'ENSG00000167912',
 'ENSG00000267197',
 'ENSG00000277715',
 'ENSG00000266709',
 'ENSG00000267106',
 'ENSG00000260597',
 'ENSG00000280798',
 'ENSG00000227438',
 'ENSG00000241772',
 'ENSG00000235023',
 'ENSG00000263766',
 'ENSG00000233930',
 'ENSG00000261578',
 'ENSG00000279663',
 'ENSG00000214248',
 'ENSG00000282304',
 'ENSG00000231689',
 'ENSG00000276174',
 'ENSG00000257698',
 'ENSG00000225880',
 'ENSG00000233393',
 'ENSG00000224635',
 'ENSG00000223797',
 'ENSG00000260803',
 'ENSG00000279555',
 'ENSG00000270115',
 'ENSG00000267172',
 'ENSG00000258017',
 'ENSG00000180539',
 'ENSG00000185065',
 'ENSG00000271382',
 'ENSG00000273015',


In [64]:
Path(pair_tsv.loc[0,1]).name.split("-sort.bed")[0]

'H3K27ac-ENSG00000255328'

In [79]:
Path(pair_tsv.loc[0,1]).name.split("-sort.bed")[0]

'H3K27ac-ENSG00000255328'

In [ ]:
GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000093100-sort__vs__H3K4me2-ENSG00000093100-sort.summary.txt

In [74]:
pair_tsv["hm-lncrna"] = pair_tsv[1].apply(lambda x: Path(x).name.split("-sort.bed")[0])
pair_tsv["rnachrom_exp-lncrna"] = pair_tsv[0].apply(lambda x: Path(x).name.split("-sort.bed")[0])

In [82]:
pair_tsv["path_to_permutation_res"] = pair_tsv["rnachrom_exp-lncrna"] + "-sort__vs__" + pair_tsv["hm-lncrna"] + "-sort.summary.txt"

In [87]:
pair_tsv.loc[0, "path_to_permutation_res"]

'HFFc6-ENSG00000255328-sort__vs__H3K27ac-ENSG00000255328-sort.summary.txt'

In [89]:
pair_tsv

,0,1,hm-lncrna,rnachtom_exp-lncrna,rnachrom_exp-lncrna,path_to_permutation_res
0,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H3K27ac-ENSG00000255328,HFFc6-ENSG00000255328,HFFc6-ENSG00000255328,HFFc6-ENSG00000255328-sort__vs__H3K27ac-ENSG00...
1,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,methylation-ENSG00000255328,HFFc6-ENSG00000255328,HFFc6-ENSG00000255328,HFFc6-ENSG00000255328-sort__vs__methylation-EN...
2,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H3K27ac-ENSG00000255328,iMARGI-HFFc6-ENSG00000255328,iMARGI-HFFc6-ENSG00000255328,iMARGI-HFFc6-ENSG00000255328-sort__vs__H3K27ac...
3,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,methylation-ENSG00000255328,iMARGI-HFFc6-ENSG00000255328,iMARGI-HFFc6-ENSG00000255328,iMARGI-HFFc6-ENSG00000255328-sort__vs__methyla...
4,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H3K27ac-ENSG00000255328,GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000255328,GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000255328,GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000255328-...
...,...,...,...,...,...,...
82780,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H4K20me1-ENSG00000224032,Red_C-fibroblasts-ENSG00000224032,Red_C-fibroblasts-ENSG00000224032,Red_C-fibroblasts-ENSG00000224032-sort__vs__H4...
82781,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H4K20me1-ENSG00000224032,iMARGI-HEK293T-ENSG00000224032,iMARGI-HEK293T-ENSG00000224032,iMARGI-HEK293T-ENSG00000224032-sort__vs__H4K20...
82782,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H4K20me1-ENSG00000224032,iMARGI-HUVEC-ENSG00000224032,iMARGI-HUVEC-ENSG00000224032,iMARGI-HUVEC-ENSG00000224032-sort__vs__H4K20me...
82783,/export/home/agletdinov/work/git_projects/fant...,/export/home/agletdinov/work/git_projects/fant...,H4K20me1-ENSG00000224032,GRID-MM_1S-ENSG00000224032,GRID-MM_1S-ENSG00000224032,GRID-MM_1S-ENSG00000224032-sort__vs__H4K20me1-...


In [155]:
def create_df_with_pval(path_to_res):
    lnc_rna = path_to_res.stem.split("__vs__")[1].split("-")[1]
    hm = path_to_res.stem.split("__vs__")[1].split("-")[0]
    p_value = pd.read_csv(path_to_res, sep="\t")["pvalue"][0]

    rnachrom_expr = "-".join(path_to_res.stem.split("__vs__")[0].split("-sort")[0].split("-")[0:-1])
    df = pd.DataFrame({"lnc_ens":[lnc_rna], "hm":[hm], "p_value":[p_value], "rnachrom_expr":[rnachrom_expr]})
    return df

In [156]:
create_df_with_pval(PERMUTATION_RES / "HFFc6-ENSG00000255328-sort__vs__H3K27ac-ENSG00000255328-sort.summary.txt")

,lnc_ens,hm,p_value,rnachrom_expr
0,ENSG00000255328,H3K27ac,0.009901,HFFc6


In [157]:
list_of_res = []
for hm_lncrna, subdf in tqdm(pair_tsv.groupby(by="hm-lncrna")):
    print(hm_lncrna)
    hm_lncrna_perm_res_concat = pd.concat([create_df_with_pval(PERMUTATION_RES / i) for i in subdf["path_to_permutation_res"]])
    
    # Применим поправку на множественные сравнения к колонке 'p_value'
    corrected = multipletests(hm_lncrna_perm_res_concat["p_value"], method="fdr_bh")
    # Добавим скорректированные p-значения в DataFrame
    hm_lncrna_perm_res_concat["p_adj"] = corrected[1]  # вторая позиция — это откорректированные p-values
    # При необходимости можно добавить столбец со значимостью
    hm_lncrna_perm_res_concat["significant"] = corrected[0]  # первая позиция — булевый массив значимости
    list_of_res.append(hm_lncrna_perm_res_concat)

  0%|                                                                                                                                                                                                                                        | 3/11451 [00:00<13:45, 13.86it/s]

H3K27ac-ENSG00000099869
H3K27ac-ENSG00000117242
H3K27ac-ENSG00000124835
H3K27ac-ENSG00000125804


  0%|▏                                                                                                                                                                                                                                       | 7/11451 [00:00<12:50, 14.85it/s]

H3K27ac-ENSG00000130600
H3K27ac-ENSG00000132204
H3K27ac-ENSG00000145075
H3K27ac-ENSG00000146521


  0%|▏                                                                                                                                                                                                                                       | 9/11451 [00:00<12:02, 15.83it/s]

H3K27ac-ENSG00000152931
H3K27ac-ENSG00000157306
H3K27ac-ENSG00000163364


... (3,619 outputs left out)

methylation-ENSG00000280800
methylation-ENSG00000281026
methylation-ENSG00000281100
methylation-ENSG00000281103
methylation-ENSG00000281181
methylation-ENSG00000281183
methylation-ENSG00000281195
methylation-ENSG00000281357
methylation-ENSG00000281398


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉| 11447/11451 [06:46<00:00, 40.71it/s]

methylation-ENSG00000281538
methylation-ENSG00000281772
methylation-ENSG00000281969
methylation-ENSG00000282057
methylation-ENSG00000282221
methylation-ENSG00000282386
methylation-ENSG00000282728
methylation-ENSG00000282772


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11451/11451 [06:46<00:00, 28.14it/s]


In [158]:
all_perm_res_adj = pd.concat(list_of_res)

In [159]:
all_perm_res_adj

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000099869,H3K27ac,1.0,HFFc6,1.0,False
0,ENSG00000099869,H3K27ac,1.0,iMARGI-HFFc6,1.0,False
0,ENSG00000099869,H3K27ac,1.0,GRID-MDA_MB_231__HTB_26_ATCC_,1.0,False
0,ENSG00000099869,H3K27ac,1.0,iMARGI-HEK293T,1.0,False
0,ENSG00000099869,H3K27ac,1.0,iMARGI-HUVEC,1.0,False
...,...,...,...,...,...,...
0,ENSG00000282772,methylation,1.0,Red_C-fibroblasts,1.0,False
0,ENSG00000282772,methylation,1.0,iMARGI-HEK293T,1.0,False
0,ENSG00000282772,methylation,1.0,iMARGI-HUVEC,1.0,False
0,ENSG00000282772,methylation,1.0,GRID-MM_1S,1.0,False


In [160]:
all_perm_res_adj.query("lnc_ens == 'ENSG00000255328' & hm == 'H3K27ac' & rnachrom_expr == 'HFFc6'")

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000255328,H3K27ac,0.009901,HFFc6,0.013861,True


In [161]:
all_perm_res_adj["p_value"] = all_perm_res_adj["p_value"].astype(float)

In [162]:
(all_perm_res_adj["p_value"] <= 1).sum()

82785

In [163]:
all_perm_res_adj.head(3)

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000099869,H3K27ac,1.0,HFFc6,1.0,False
0,ENSG00000099869,H3K27ac,1.0,iMARGI-HFFc6,1.0,False
0,ENSG00000099869,H3K27ac,1.0,GRID-MDA_MB_231__HTB_26_ATCC_,1.0,False


In [154]:
to_file = FANTOM / "fantom_lv2-hg38-ens_name-mapping.tsv.gz"
fantom_lv2_mapping_hg38 = pd.read_csv(to_file, sep="\t", index_col="geneID_without_v")
fantom_lv2_mapping_hg38.head()

,geneName
geneID_without_v,
CATG00000000002,CATG00000000002.1
CATG00000000003,CATG00000000003.1
CATG00000000004,CATG00000000004.1
CATG00000000005,CATG00000000005.1
CATG00000000006,CATG00000000006.1


In [179]:
mask =  fantom_lv2_mapping_hg38["geneName"].apply(lambda x: "AC016026" in x)
mask.sum()

0

In [173]:
len(set(all_perm_res_adj["lnc_ens"].unique()) - set(fantom_lv2_mapping_hg38.index))

803

In [175]:
(set(all_perm_res_adj["lnc_ens"].unique()) - set(fantom_lv2_mapping_hg38.index))

{'ENSG00000093100',
 'ENSG00000125804',
 'ENSG00000149656',
 'ENSG00000177234',
 'ENSG00000178440',
 'ENSG00000179577',
 'ENSG00000182366',
 'ENSG00000204528',
 'ENSG00000205611',
 'ENSG00000206195',
 'ENSG00000215241',
 'ENSG00000215244',
 'ENSG00000218537',
 'ENSG00000223685',
 'ENSG00000224216',
 'ENSG00000224714',
 'ENSG00000224885',
 'ENSG00000224958',
 'ENSG00000225163',
 'ENSG00000225742',
 'ENSG00000225880',
 'ENSG00000225948',
 'ENSG00000226067',
 'ENSG00000226276',
 'ENSG00000226383',
 'ENSG00000226476',
 'ENSG00000226578',
 'ENSG00000226777',
 'ENSG00000226854',
 'ENSG00000227189',
 'ENSG00000227192',
 'ENSG00000227388',
 'ENSG00000227634',
 'ENSG00000227733',
 'ENSG00000227888',
 'ENSG00000228203',
 'ENSG00000228423',
 'ENSG00000228434',
 'ENSG00000229019',
 'ENSG00000229236',
 'ENSG00000229404',
 'ENSG00000229425',
 'ENSG00000230415',
 'ENSG00000230537',
 'ENSG00000230896',
 'ENSG00000230928',
 'ENSG00000231305',
 'ENSG00000231527',
 'ENSG00000231646',
 'ENSG00000232117',


In [169]:
all_perm_res_adj["lnc_ens"].unique()

array(['ENSG00000099869', 'ENSG00000117242', 'ENSG00000124835', ...,
       'ENSG00000279789', 'ENSG00000280232', 'ENSG00000280321'],
      dtype=object)

In [180]:
all_perm_res_adj["lnc"] = all_perm_res_adj["lnc_ens"].map(
    fantom_lv2_mapping_hg38["geneName"]
)

In [182]:
all_perm_res_adj["lnc_ens"].nunique(), all_perm_res_adj["lnc"].nunique()

(3182, 2379)

In [186]:
to_save = PERMUTATION_RES_AFTER_FDR / "permutation_fdr_concat.tsv"
all_perm_res_adj.to_csv(to_save, sep="\t", index=False)

In [183]:
mask = all_perm_res_adj["p_adj"] <= 0.05
all_perm_res_adj_signif = all_perm_res_adj[mask]
all_perm_res_adj_signif

,lnc_ens,hm,p_value,rnachrom_expr,p_adj,significant,lnc
0,ENSG00000130600,H3K27ac,0.009901,HFFc6,0.017327,True,H19
0,ENSG00000130600,H3K27ac,0.009901,iMARGI-HFFc6,0.017327,True,H19
0,ENSG00000130600,H3K27ac,0.009901,iMARGI-HEK293T,0.017327,True,H19
0,ENSG00000130600,H3K27ac,0.009901,iMARGI-HUVEC,0.017327,True,H19
0,ENSG00000130600,H3K27ac,0.019802,GRID-MM_1S,0.027723,True,H19
...,...,...,...,...,...,...,...
0,ENSG00000281398,methylation,0.009901,Red_C-K562,0.023102,True,NaN
0,ENSG00000282057,methylation,0.009901,HFFc6,0.009901,True,NaN
0,ENSG00000282057,methylation,0.009901,iMARGI-HFFc6,0.009901,True,NaN
0,ENSG00000282057,methylation,0.009901,GRID-MDA_MB_231__HTB_26_ATCC_,0.009901,True,NaN


In [184]:
all_perm_res_adj_signif["lnc_ens"].nunique(), all_perm_res_adj_signif["lnc"].nunique()

(919, 756)

In [146]:
all_perm_res_adj[mask]["lnc_rna"].nunique()

919

In [187]:
to_save = PERMUTATION_RES_AFTER_FDR / "permutation_fdr_concat-signif.tsv"
all_perm_res_adj_signif.to_csv(to_save, sep="\t", index=False)

In [10]:
list_of_df = defaultdict(list)
for i in PERMUTATION_RES.glob("*.txt*"):
    lnc_rna = i.stem.split("__vs__")[1].split("-")[1]
    hm = i.stem.split("__vs__")[1].split("-")[0]
    p_value = pd.read_csv(i, sep="\t")["pvalue"][0]

    rnachrom_expr = "-".join(i.stem.split("__vs__")[0].split("-sort")[0].split("-")[0:-1])
    df = pd.DataFrame({"lnc_rna":[lnc_rna], "hm":[hm], "p_value":[p_value], "rnachrom_expr":[rnachrom_expr]})



    # Применим поправку на множественные сравнения к колонке 'p_value'
    corrected = multipletests(df["p_value"], method="fdr_bh")
    
    # Добавим скорректированные p-значения в DataFrame
    df["p_adj"] = corrected[1]  # вторая позиция — это откорректированные p-values
    
    # При необходимости можно добавить столбец со значимостью
    df["significant"] = corrected[0]  # первая позиция — булевый массив значимости
    
    list_of_df[rnachrom_expr].append(df)

In [11]:
len(list_of_df)

8

In [12]:
for i in list_of_df.keys():
    list_of_df[i] = pd.concat(list_of_df[i])

In [14]:
list_of_df.keys()

dict_keys(['GRID-MDA_MB_231__HTB_26_ATCC_', 'iMARGI-HEK293T', 'iMARGI-HUVEC', 'HFFc6', 'iMARGI-HFFc6', 'GRID-MM_1S', 'Red_C-K562', 'Red_C-fibroblasts'])

In [18]:
list_of_df["Red_C-fibroblasts"]

,lnc_rna,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000267838,H3K27ac,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000267838,H3K9ac,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000267838,methylation,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000093100,H3K4me2,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000093100,H3K79me2,0.940594,Red_C-fibroblasts,0.940594,False
...,...,...,...,...,...,...
0,ENSG00000236333,methylation,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000236333,H4K20me1,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000260830,H3K9ac,1.000000,Red_C-fibroblasts,1.000000,False
0,ENSG00000260830,methylation,1.000000,Red_C-fibroblasts,1.000000,False


In [245]:
for i in list_of_df.keys():
    a = list_of_df[i][list_of_df[i]["significant"] == "True"]
    break

In [246]:
a

,lnc_rna,hm,p_value,rnachrom_expr,p_adj,significant


In [206]:
res_df = pd.concat(list_of_df)

In [207]:
import pandas as pd
from statsmodels.stats.multitest import multipletests

# Применим поправку на множественные сравнения к колонке 'p_value'
corrected = multipletests(res_df["p_value"], method="fdr_bh")

# Добавим скорректированные p-значения в DataFrame
res_df["p_adj"] = corrected[1]  # вторая позиция — это откорректированные p-values

# При необходимости можно добавить столбец со значимостью
res_df["significant"] = corrected[0]  # первая позиция — булевый массив значимости

In [208]:
res_df

,lnc_rna,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000282057,H3K9ac,0.009901,GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057,0.242144,False
0,ENSG00000255328,H3K27ac,0.950495,iMARGI-HEK293T-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,iMARGI-HEK293T-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,iMARGI-HUVEC-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,HFFc6-ENSG00000255328,1.000000,False
...,...,...,...,...,...,...
0,ENSG00000265369,methylation,1.000000,HFFc6-ENSG00000265369,1.000000,False
0,ENSG00000265369,methylation,1.000000,iMARGI-HFFc6-ENSG00000265369,1.000000,False
0,ENSG00000265369,H3K27ac,1.000000,iMARGI-HEK293T-ENSG00000265369,1.000000,False
0,ENSG00000265369,H3K27ac,1.000000,iMARGI-HUVEC-ENSG00000265369,1.000000,False


In [209]:
res_df

,lnc_rna,hm,p_value,rnachrom_expr,p_adj,significant
0,ENSG00000282057,H3K9ac,0.009901,GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057,0.242144,False
0,ENSG00000255328,H3K27ac,0.950495,iMARGI-HEK293T-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,iMARGI-HEK293T-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,iMARGI-HUVEC-ENSG00000255328,1.000000,False
0,ENSG00000255328,methylation,1.000000,HFFc6-ENSG00000255328,1.000000,False
...,...,...,...,...,...,...
0,ENSG00000265369,methylation,1.000000,HFFc6-ENSG00000265369,1.000000,False
0,ENSG00000265369,methylation,1.000000,iMARGI-HFFc6-ENSG00000265369,1.000000,False
0,ENSG00000265369,H3K27ac,1.000000,iMARGI-HEK293T-ENSG00000265369,1.000000,False
0,ENSG00000265369,H3K27ac,1.000000,iMARGI-HUVEC-ENSG00000265369,1.000000,False


In [210]:
res_df["p_adj"].min()

0.24214377959535077

In [148]:
i.stem.split("__vs__")[1].split("-")[0]

'H3K9ac'

In [149]:
i.stem.split("__vs__")[1].split("-")[1]

'ENSG00000282057'

In [151]:
i.stem.split("__vs__")[0].split("-sort")[0]

'GRID-MDA_MB_231__HTB_26_ATCC_-ENSG00000282057'